# WeightLens and CircuitLens on Gemma-2-2B

This notebook reruns feature **layer 0 / index 24** from *Circuit Insights: Towards Interpretability Beyond Activations*. It applies TDHook's public WeightLens projection and CircuitLens attention decomposition to the published Gemma-2-2B replacement model and Gemma Scope transcoders, then compares the selected tokens and contributors with the published records.

It needs a Hugging Face account with access to Gemma, about 20 GB of downloads, and 16 GB of accelerator memory.

## Setup

Run from a TDHook checkout with `uv run --with jupyterlab --extra circuit-lens --extra circuit-clustering --group notebooks jupyter lab`. The reference file contains the first 12 published examples for this feature and the revisions they came from.

In [1]:
from __future__ import annotations

import importlib.metadata
import json
import os
import random
import subprocess
from pathlib import Path

import numpy as np

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch
from circuit_tracer import ReplacementModel
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tdhook.attribution import (
    CircuitLensArtifact,
    attention_contributions,
    cluster_circuit_artifacts,
)
from tdhook.weights import analyze_input_invariant_feature, select_projection_outliers

SEED = 127
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

if not torch.cuda.is_available():
    raise RuntimeError("This exact Gemma-2-2B reproduction requires a CUDA accelerator with about 16 GB memory.")

device = torch.device("cuda")
dtype = torch.bfloat16
reference_relative_path = Path("docs/source/notebooks/assets/gemma-2-2b-feature-24-reference.json")
reference_path = next(
    (
        root / reference_relative_path
        for root in (Path.cwd(), *Path.cwd().parents)
        if (root / reference_relative_path).is_file()
    ),
    None,
)
if reference_path is None:
    raise FileNotFoundError(f"Could not find {reference_relative_path} from {Path.cwd()} or its parents")
result_path = reference_path.with_name("gemma-2-2b-feature-24-results.json")
reference = json.loads(reference_path.read_text())
feature_layer = reference["feature"]["layer"]
feature_index = reference["feature"]["index"]
reference["provenance"]

{'model': 'google/gemma-2-2b',
 'model_revision': 'c5ebcd40d208330abc697524c919956e692655cf',
 'transcoder_set': 'mwhanna/gemma-scope-transcoders',
 'transcoder_revision': 'bd5773156dea09893636c801df1237d0410307d2',
 'circuit_dataset': 'egolimblevskaia/circuitlens-gemma-2-2b-transcoder-circuit-analysis',
 'circuit_dataset_revision': 'fcd78fd98c3de6cc869d7df9e7d0f4a864ffea50',
 'circuit_file': 'analysis_layer_0.jsonl',
 'weight_dataset': 'egolimblevskaia/weightlens-gemma-2-2b-transcoder-descriptions',
 'weight_dataset_revision': 'aa51f4ee40784ac7af168932e2505fe3e4bab833',
 'weight_file': 'feature_analysis_layer_0.json',
 'weightlens_revision': '93f820024034bb9b1829f7f09c1483ec3bc71f49',
 'circuitlens_revision': 'd24b7e3a71ea1fbce0800056eb7333d1a282303d'}

In [2]:
model_snapshot = snapshot_download(
    reference["provenance"]["model"],
    revision=reference["provenance"]["model_revision"],
    allow_patterns=[
        "config.json",
        "special_tokens_map.json",
        "tokenizer.json",
        "tokenizer.model",
        "tokenizer_config.json",
    ],
)
tokenizer = AutoTokenizer.from_pretrained(model_snapshot)

model = ReplacementModel.from_pretrained(
    reference["provenance"]["model"],
    f"{reference['provenance']['transcoder_set']}@{reference['provenance']['transcoder_revision']}",
    revision=reference["provenance"]["model_revision"],
    device=device,
    dtype=dtype,
    tokenizer=tokenizer,
)
model.eval()

selected_transcoder = model.transcoders[feature_layer]


def encoder_for_tdhook(transcoder):
    """Normalize circuit-tracer releases to TDHook's [model, feature] convention."""
    weights = transcoder.W_enc
    if weights.shape[0] == model.cfg.d_model:
        return weights
    if weights.shape[1] == model.cfg.d_model:
        return weights.T
    raise ValueError(f"Unrecognized encoder shape: {tuple(weights.shape)}")


# The selected feature is in layer 0, so WeightLens needs no earlier dictionaries.
# Keeping one layer also avoids materializing every lazily loaded decoder.
feature_encoders = (encoder_for_tdhook(selected_transcoder),)
feature_decoders = (selected_transcoder.W_dec,)
token_labels = tuple(model.tokenizer.decode([index]) for index in range(model.cfg.d_vocab))

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


## WeightLens

TDHook projects the feature encoder into the embedding dictionary and its decoder into the unembedding dictionary. We use the candidate pooling and z-score settings from WeightLens.

In [3]:
weight_artifact = analyze_input_invariant_feature(
    feature_layer=feature_layer,
    feature_index=feature_index,
    embedding=model.W_E,
    unembedding=model.W_U,
    feature_encoders=feature_encoders,
    feature_decoders=feature_decoders,
    token_outlier_threshold=5.5,
    feature_outlier_threshold=4.0,
    token_pool_size=1000,
    feature_pool_size=100,
    input_token_labels=token_labels,
    output_token_labels=token_labels,
)

expected_positive = [item["token_id"] for item in reference["weightlens"]["embedding_positive"]]
expected_negative = [item["token_id"] for item in reference["weightlens"]["embedding_negative"]]
observed_positive = [item.index for item in weight_artifact.embedding_positive]
observed_negative = [item.index for item in weight_artifact.embedding_negative]
observed_output_labels = [item.label for item in weight_artifact.output_positive]

{
    "embedding_positive": [(item.index, item.label, item.score) for item in weight_artifact.embedding_positive],
    "embedding_negative": [(item.index, item.label, item.score) for item in weight_artifact.embedding_negative],
    "output_positive": [(item.index, item.label, item.score) for item in weight_artifact.output_positive],
}

{'embedding_positive': [(19538, ' accused', 22.875), (4818, ' saw', 21.5)],
 'embedding_negative': [(115269, ' InputDecoration', -35.0),
  (143572, 'RegistryLite', -34.25)],
 'output_positive': [(185265, 'tvguidetime', 0.345703125),
  (149927, ' HasFactory', 0.3125)]}

## CircuitLens

For each published input, TDHook decomposes the layer-0 attention output with the observed attention pattern and the target feature encoder. The selected head and source-token pairs are compared directly with the published artifact.

In [4]:
pattern_name = "blocks.0.attn.hook_pattern"
value_name = "blocks.0.attn.hook_v"
feature_input_name = f"blocks.0.{model.feature_input_hook}"
names = {pattern_name, value_name, feature_input_name}
encoder = feature_encoders[feature_layer][:, feature_index].detach().float().cpu()
output_weight = model.blocks[0].attn.W_O.detach().float().cpu()

circuit_artifacts = []
score_deltas = []
activation_deltas = []
contributor_identity_matches = []
contributor_comparisons = []

for sample in reference["samples"]:
    tokens = torch.tensor(sample["input_ids"], device=device).unsqueeze(0)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=lambda name: name in names, stop_at_layer=1)

    pattern = cache[pattern_name][0].detach().float().cpu()
    values = cache[value_name][0].detach().float().cpu()
    if values.shape[1] != pattern.shape[0]:
        values = values.repeat_interleave(pattern.shape[0] // values.shape[1], dim=1)

    position = sample["target_position"]
    output_gradient = torch.zeros(pattern.shape[1], model.cfg.d_model)
    output_gradient[position] = encoder
    all_attention = attention_contributions(
        pattern,
        values,
        output_weight,
        output_gradient,
        layer=0,
        target_position=position,
    )
    selected_positions = select_projection_outliers(
        torch.tensor([item.score for item in all_attention]),
        threshold=5.0,
        largest=True,
    )
    selected = tuple(all_attention[item.index] for item in selected_positions)

    observed_ids = {(item.head_index, item.source_token) for item in selected}
    expected_ids = {(item["head"], item["source_token"]) for item in sample["attention"]}
    contributor_identity_matches.append(observed_ids == expected_ids)
    contributor_comparisons.append(
        {
            "published": sorted([head, token] for head, token in expected_ids),
            "observed": sorted([head, token] for head, token in observed_ids),
        }
    )
    observed_by_id = {(item.head_index, item.source_token): item.score for item in selected}
    score_deltas.extend(
        abs(observed_by_id[key] - item["score"])
        for item in sample["attention"]
        if (key := (item["head"], item["source_token"])) in observed_by_id
    )
    feature_input = cache[feature_input_name][0, position].detach()
    target_activation = float(selected_transcoder.encode(feature_input)[feature_index].float().cpu())
    activation_deltas.append(abs(target_activation - sample["target_activation"]))
    circuit_artifacts.append(
        CircuitLensArtifact(
            target_layer=feature_layer,
            target_feature_index=feature_index,
            target_position=(position,),
            target_activation=target_activation,
            upstream_features=(),
            attention=selected,
            output_logits=(),
        )
    )

discrepancies = {
    "max_attention_score_abs_delta": max(score_deltas, default=0.0),
    "max_target_activation_abs_delta": max(activation_deltas, default=0.0),
}
discrepancies

{'max_attention_score_abs_delta': 0.0043784379959106445,
 'max_target_activation_abs_delta': 0.0625}

## Circuit clustering

The 12 circuits are also clustered by their relative-token signatures. The paper fitted clusters on 100 examples, so this small run simply shows the public clustering API rather than comparing cluster labels.

In [5]:
clusters = cluster_circuit_artifacts(
    circuit_artifacts,
    min_frequency=0.05,
    min_abs_score=0.0,
    eps=0.8,
    min_samples=2,
)

bounded_cluster_report = {
    "examples": len(circuit_artifacts),
    "labels": clusters.labels,
    "reference_full_sample_labels_for_same_inputs": tuple(item["reference_label"] for item in reference["samples"]),
    "nonempty_signatures": sum(bool(item) for item in clusters.filtered_signatures),
}
bounded_cluster_report

{'examples': 12,
 'labels': (0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1),
 'reference_full_sample_labels_for_same_inputs': (-1,
  -1,
  0,
  0,
  -1,
  -1,
  0,
  0,
  0,
  0,
  0,
  0),
 'nonempty_signatures': 8}

## Results

The saved result records the observed matches and numerical differences for this run. It covers one feature and 12 of the 100 published CircuitLens examples.

In [6]:
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


run_record = {
    "weightlens": {
        "published": {
            "embedding_positive": expected_positive,
            "embedding_negative": expected_negative,
            "output_positive_labels": reference["weightlens"]["output_positive_labels"],
        },
        "observed": {
            "embedding_positive": observed_positive,
            "embedding_negative": observed_negative,
            "output_positive_labels": observed_output_labels,
        },
    },
    "circuitlens": {
        "matching_examples": sum(contributor_identity_matches),
        "examples": len(contributor_identity_matches),
        "numerical_differences": discrepancies,
        "comparisons": contributor_comparisons,
    },
    "model": reference["provenance"]["model"],
    "transcoder_set": reference["provenance"]["transcoder_set"],
    "feature": reference["feature"],
    "sample_count": len(reference["samples"]),
    "seed": SEED,
    "device": str(device),
    "device_name": torch.cuda.get_device_name(device),
    "dtype": str(dtype),
    "torch": torch.__version__,
    "tdhook": package_version("tdhook"),
    "circuit_tracer": package_version("circuit-tracer"),
    "transformer_lens": package_version("transformer-lens"),
    "scikit_learn": package_version("scikit-learn"),
    "tdhook_revision": subprocess.run(
        ["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True
    ).stdout.strip(),
    "reference": reference["provenance"],
    "numerical_discrepancies": discrepancies,
    "cluster_report": bounded_cluster_report,
    "scope": [
        "12 of 100 published examples for one feature",
        "layer 0 has no upstream transcoder-feature contributors",
        "no 24M-token activation collection",
        "no paper-wide table, description, evaluation, or causal-effect reproduction",
    ],
}
result_text = json.dumps(run_record, indent=2, sort_keys=True, default=list) + "\n"
result_path.write_text(result_text)
print(result_text)

{
  "circuit_tracer": "0.5.0",
  "circuitlens": {
    "comparisons": [
      {
        "observed": [],
        "published": []
      },
      {
        "observed": [],
        "published": []
      },
      {
        "observed": [
          [
            0,
            11
          ],
          [
            5,
            11
          ]
        ],
        "published": [
          [
            0,
            11
          ],
          [
            5,
            11
          ]
        ]
      },
      {
        "observed": [
          [
            0,
            41
          ],
          [
            4,
            41
          ]
        ],
        "published": [
          [
            0,
            41
          ],
          [
            4,
            41
          ]
        ]
      },
      {
        "observed": [],
        "published": []
      },
      {
        "observed": [],
        "published": []
      },
      {
        "observed": [
          [
            4,
          